In [1]:
from PIL import Image
import numpy as np
import os, sys
import glob
import matplotlib.pyplot as plt
from src import data_utils

In [4]:
image_paths = glob.glob(os.path.join("data", "bscans","ORIGINAL", "*", "*.png"))
mask1_paths = glob.glob(os.path.join("data", "bscans","RPE", "*", "*.png"))
mask2_paths = glob.glob(os.path.join("data", "bscans","ILM", "*", "*.png"))

image_paths = sorted(image_paths)
mask1_paths = sorted(mask1_paths)
mask2_paths = sorted(mask2_paths)

In [5]:
assert len(mask1_paths) == len(mask2_paths)
sss2 = 0
sss1 = 0

for mask_idx in range(len(mask1_paths)):
    mask1 = Image.open(mask1_paths[mask_idx])
    mask1 = np.array(mask1)
    mask1 = np.expand_dims(mask1, axis=-1)

    mask2 = Image.open(mask2_paths[mask_idx])
    mask2 = np.array(mask2)
    mask2 = np.expand_dims(mask2, axis=-1)

    assert mask1.shape == mask2.shape

    mask1[mask1 >= 1] = 1
    mask1[mask1 < 1] = 0
    mask1[mask2 >= 1] = 2
    sss2 += np.sum(mask1== 2)
    sss1 += np.sum(mask1== 1)

    mask_save_path = mask1_paths[mask_idx].replace("bscans", "bscans-derived")
    mask_save_path = mask_save_path.replace(".png", ".npy")
    if not os.path.exists(os.path.dirname(mask_save_path)):
        os.makedirs(os.path.dirname(mask_save_path)) 
    
    np.save(mask_save_path, mask1)

print(sss1)
print(sss2)

for image_path in image_paths:
    image = Image.open(image_path)
    image = np.array(image)
    image = np.expand_dims(image, axis=-1)

    image_save_path = image_path.replace("bscans", "bscans-derived")
    image_save_path = image_save_path.replace(".png", ".npy")
    if not os.path.exists(os.path.dirname(image_save_path)):
        os.makedirs(os.path.dirname(image_save_path)) 
    np.save(image_save_path, image)

285639
585873


In [6]:
image_save_path

'data/bscans-derived/ORIGINAL/2/BSCAN_58524952_2.npy'

In [9]:
train_image_paths = []
val_image_paths = []
train_mask_paths = []
val_mask_paths = []

for image_path in glob.glob(os.path.join("data", "bscans-derived","ORIGINAL", "*", "*.npy")):
    if image_path.split(os.path.sep)[3] in ["1"]:
        train_image_paths.append(image_path)
    else:
        val_image_paths.append(image_path)

for mask_path in glob.glob(os.path.join("data", "bscans-derived","RPE", "*", "*.npy")):
    if mask_path.split(os.path.sep)[3] in ["1"]:
        train_mask_paths.append(mask_path)
    else:
        val_mask_paths.append(mask_path)

data_utils.write_paths(os.path.join("training", "bscans-train-images.txt"), train_image_paths)
data_utils.write_paths(os.path.join("training", "bscans-train-masks.txt"), train_mask_paths)
data_utils.write_paths(os.path.join("validation", "bscans-val-images.txt"), val_image_paths)
data_utils.write_paths(os.path.join("validation", "bscans-val-masks.txt"), val_mask_paths)

In [10]:
all_images = []
for image_path in image_paths:
    image = Image.open(image_path)
    image = np.array(image).astype(np.float32)

    all_images.append(image)
    
all_images = np.stack(all_images, 0)
print(all_images.shape)
print(np.mean(all_images))
print(np.std(all_images))

    

(331, 1024, 400)
49.757008
27.255404
